# Chapter 7 — Batch API and Cost Optimization · Practice

**Goal:** put a number on every cost decision. Build and run a Batch API pipeline end to end (JSONL, `custom_id`s, output and error files, resubmission), choose a processing tier, structure prompts for caching, plan rate limits and retries, and model cost per request, per workload and per successful task.

- Theory: [README.md](README.md) (sections 7.1–7.12). This notebook holds the code.
- **Estimated time:** 3–4 hours
- **Estimated API cost:** **< $0.02** with the default `gpt-6-luna` if you run every `# (live)` cell (a few short requests, one cache demo with a ~1,500-token prefix, and an optional 3-line batch). Everything else is free.
- **Prerequisites:** [00 — Prerequisites](../../00-prerequisites/README.md) (venv, `.env` with `OPENAI_API_KEY`), chapters [01](../01-responses-api-fundamentals/README.md), [02](../02-function-calling-and-structured-outputs/README.md) and [06](../06-prompt-engineering-and-evals/README.md) of this path.

> **No API key or no credit?** Skip the key-check cell and every cell whose first line is `# (live)`. All other cells, including the whole mini-project, run offline: a small **fake OpenAI server** answers the *real* `openai` SDK through `httpx2.MockTransport`, so you exercise the SDK's own request building, response types, error classes and retry logic without a network call.

In [ ]:
import base64
import email.utils
import hashlib
import hmac
import json
import math
import os
import random
import re
import tempfile
import time
import typing
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta, timezone
from email.parser import BytesParser
from email.policy import HTTP
from pathlib import Path

import httpx2
from dotenv import find_dotenv, load_dotenv
from openai import APIConnectionError, APIStatusError, APITimeoutError, InternalServerError, OpenAI, RateLimitError
from openai.types.responses import Response, ResponseUsage
from pydantic import BaseModel, TypeAdapter, ValidationError

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")       # cheapest GPT-6 model; the right default for bulk extraction
WORKDIR = Path(tempfile.mkdtemp(prefix="oa_ch07_"))    # JSONL files and manifests go here, never into the repo
LIVE_MARKER = Path(tempfile.gettempdir()) / "oa_ch07_live_batch_id.txt"   # remembers the optional live batch
print("model:", MODEL, "| scratch dir:", WORKDIR)

In [ ]:
# Live cells need a key. Offline cells keep working if you skip this one.
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see ../../00-prerequisites/README.md)"
client = OpenAI()

### Shared offline helpers

Run the next cell once. It defines `FakeOpenAI`, a deterministic stand-in for the Files, Batches and Responses endpoints, and `offline_client(backend)`, which returns a **real** `OpenAI` client wired to it. The fake "model" extracts invoice fields with a regex from the **user's document only** (never from the rules or worked examples in the instructions), returns all-`null` fields when the document is not an invoice, and markers in a document trigger the failure layers from [7.4](README.md#74-reading-results-and-handling-failures): `[[FLAKY]]` (a `500` on the first attempt), `[[RAMBLING]]` (`incomplete` when `max_output_tokens` is small), `[[SMUDGED]]` (a currency symbol that passes the schema but fails a business rule), and oversized documents (a `400` over a deliberately tiny context window). Each `batches.retrieve` call advances a batch by one step, so "24 hours" takes a few polls.

In [ ]:
# FakeOpenAI: a deterministic, offline stand-in for api.openai.com (Files, Batches, Responses).
# The REAL openai SDK talks to it through httpx2.MockTransport, so request building, response
# parsing, error classes and SDK retries are the SDK's own code. Only the server is fake.
# Comments starting with "fake:" mark simplifications that are NOT documented OpenAI behavior.
FAKE_CONTEXT_LIMIT = 3_000   # fake: a tiny "context window" (tokens) so an oversized document fails with a 400
NOW = 1_790_000_000          # fixed timestamp keeps every run identical


def rough_tokens(text: str) -> int:
    """~4 characters per token. Fine for planning; use client.responses.input_tokens.count for real numbers."""
    return max(1, len(text) // 4)


def api_error(message: str, type_: str, code: str | None = None, param: str | None = None) -> dict:
    return {"error": {"message": message, "type": type_, "param": param, "code": code}}


def response_json(model: str, text: str | None, *, input_tokens: int, n: int = 0, status: str = "completed",
                  reason: str | None = None, reasoning: int = 64, cached: int = 0, written: int = 0,
                  service_tier: str = "default") -> dict:
    """A Responses API body, validated by the SDK's own Response model so the shape is real."""
    output = [] if text is None else [{
        "type": "message", "id": f"msg_{n:05d}", "role": "assistant", "status": "completed",
        "content": [{"type": "output_text", "text": text, "annotations": []}]}]
    out_tokens = reasoning + (rough_tokens(text) if text else 0)
    data = {
        "id": f"resp_{n:05d}", "object": "response", "created_at": NOW, "model": model, "status": status,
        "incomplete_details": {"reason": reason} if reason else None, "output": output,
        "parallel_tool_calls": True, "tool_choice": "auto", "tools": [], "service_tier": service_tier,
        "usage": {"input_tokens": input_tokens,
                  "input_tokens_details": {"cached_tokens": cached, "cache_write_tokens": written},
                  "output_tokens": out_tokens, "output_tokens_details": {"reasoning_tokens": reasoning},
                  "total_tokens": input_tokens + out_tokens},
    }
    return Response.model_validate(data).model_dump(mode="json", exclude_none=True)


DOC_RE = re.compile(r"Vendor: (?P<vendor>[^|]+?) \| Date: (?P<date>[^|]+?) \| "
                    r"Currency: (?P<currency>[^|]+?) \| Total: (?P<total>[\d.]+)")


def document_of(body: dict) -> str:
    """fake: the per-request document is the USER content only. Instructions and developer messages hold rules and
    worked examples that look like invoices, so the fake 'model' must never extract from them."""
    raw_input = body.get("input")
    if isinstance(raw_input, str):
        return raw_input
    parts = []
    for item in raw_input or []:
        if item.get("role") != "user":
            continue
        content = item.get("content")
        if isinstance(content, str):
            parts.append(content)
        else:
            parts += [c.get("text", "") for c in content or [] if c.get("type") == "input_text"]
    return "\n".join(parts)


def simulated_model(body: dict, attempt: int, n: int = 0) -> tuple[int, dict]:
    """fake: a deterministic 'model'. Markers in a document trigger the failure layers from README 7.4."""
    raw_input = body.get("input")
    prompt = (body.get("instructions") or "") + (raw_input if isinstance(raw_input, str) else json.dumps(raw_input))
    document = document_of(body)                                          # extract from the document, not the rules
    model, tokens_in = body.get("model", "?"), rough_tokens(prompt)
    if tokens_in > FAKE_CONTEXT_LIMIT:                                    # invalid request: fix, don't resubmit
        return 400, api_error(f"Input is {tokens_in} tokens, over the context window.",
                              "invalid_request_error", "context_length_exceeded", "input")  # fake: illustrative code
    if "[[FLAKY]]" in document and attempt == 1:                          # transient: resubmit as-is
        return 500, api_error("The server had an error while processing your request.", "server_error")
    if "[[RAMBLING]]" in document and body.get("max_output_tokens", 10**6) < 4000:   # 200 but incomplete
        return 200, response_json(model, None, input_tokens=tokens_in, n=n, status="incomplete",
                                  reason="max_output_tokens", reasoning=body["max_output_tokens"])
    match = DOC_RE.search(document)
    if not match:                                                         # 200, schema-valid, but not an invoice
        empty = {"vendor": None, "invoice_date": None, "currency": None, "total": None}
        return 200, response_json(model, json.dumps(empty), input_tokens=tokens_in, n=n)
    f = match.groupdict()
    currency = "€" if "[[SMUDGED]]" in document else f["currency"].strip()  # schema-valid, business-rule-invalid
    data = {"vendor": f["vendor"].strip(), "invoice_date": f["date"].strip(), "currency": currency,
            "total": float(f["total"])}
    return 200, response_json(model, json.dumps(data), input_tokens=tokens_in, n=n)


class FakeOpenAI:
    """Routes: POST /files, GET /files/{id}/content, POST /batches, GET /batches/{id},
    POST /batches/{id}/cancel, POST /responses. Each GET /batches/{id} advances the batch one step."""

    def __init__(self, *, lines_per_poll: int = 50, window_polls: int = 10, seed: int = 7,
                 fast_downgrade: bool = False, rpm: int = 5_000, tpm: int = 2_000_000, project_tier: str = "default",
                 allowed_tiers: set[str] | None = None):
        self.files: dict[str, bytes] = {}
        self.batches: dict[str, dict] = {}
        self.script: list[tuple[int, dict, dict]] = []   # queued ERROR replies (status, body, headers) for POST /responses
        self.log: list[str] = []                          # one entry per HTTP request the SDK actually sent
        self.attempts: Counter = Counter()                # how often the "model" has seen each request
        self.rng = random.Random(seed)
        self.lines_per_poll, self.window_polls = lines_per_poll, window_polls
        self.fast_downgrade, self.rpm, self.tpm = fast_downgrade, rpm, tpm
        self.project_tier = project_tier                  # "Project Service Tier" setting: used when service_tier is omitted or "auto"
        self.flex_unavailable = 0                         # how many upcoming FLEX requests get "429 Resource Unavailable"
        self.allowed_tiers = allowed_tiers                # project's allowed service tiers (None = no restriction)
        self.used_requests = self.used_tokens = 0
        self.n = 0

    # -- plumbing -------------------------------------------------------------------------------
    def _id(self, prefix: str) -> str:
        self.n += 1
        return f"{prefix}_{self.n:05d}"

    def handle(self, request: httpx2.Request) -> httpx2.Response:
        request.read()
        path = request.url.path.removeprefix("/v1")
        self.log.append(f"{request.method} {path}")
        if request.method == "POST" and path == "/files":
            return self._upload(request)
        if m := re.fullmatch(r"/files/([\w-]+)/content", path):
            return httpx2.Response(200, content=self.files[m[1]])
        if request.method == "POST" and path == "/batches":
            return self._create_batch(json.loads(request.content))
        if m := re.fullmatch(r"/batches/([\w-]+)/cancel", path):
            self.batches[m[1]]["status"] = "cancelling"
            return httpx2.Response(200, json=self._public(self.batches[m[1]]))
        if m := re.fullmatch(r"/batches/([\w-]+)", path):
            return httpx2.Response(200, json=self._public(self._advance(self.batches[m[1]])))
        if request.method == "POST" and path == "/responses":
            return self._respond(json.loads(request.content))
        return httpx2.Response(404, json=api_error(f"fake: no route for {path}", "invalid_request_error"))

    @staticmethod
    def _public(b: dict) -> dict:
        return {k: v for k, v in b.items() if not k.startswith("_")}

    def _store(self, records: list[dict]) -> str | None:
        if not records:
            return None
        fid = self._id("file")
        self.files[fid] = "".join(json.dumps(r) + "\n" for r in records).encode()
        return fid

    # -- files ----------------------------------------------------------------------------------
    def _upload(self, request: httpx2.Request) -> httpx2.Response:
        raw = b"Content-Type: " + request.headers["content-type"].encode() + b"\r\n\r\n" + request.content
        parts = {p.get_param("name", header="content-disposition"): p
                 for p in BytesParser(policy=HTTP).parsebytes(raw).iter_parts()}
        content = parts["file"].get_payload(decode=True)
        fid = self._id("file")
        self.files[fid] = content
        return httpx2.Response(200, json={
            "id": fid, "object": "file", "bytes": len(content), "created_at": NOW, "status": "processed",
            "filename": parts["file"].get_filename() or "input.jsonl", "purpose": parts["purpose"].get_content().strip()})

    # -- batches --------------------------------------------------------------------------------
    def _create_batch(self, params: dict) -> httpx2.Response:
        lines, errors, seen, models = [], [], set(), set()
        for i, raw in enumerate(self.files[params["input_file_id"]].decode().splitlines(), start=1):
            if not raw.strip():
                continue
            try:
                line = json.loads(raw)
            except json.JSONDecodeError:
                errors.append(("invalid_json_line", i, "Line is not valid JSON.", None))   # fake: illustrative codes
                continue
            if line.get("custom_id") in seen:
                errors.append(("duplicate_custom_id", i, f"Duplicate custom_id {line.get('custom_id')!r}.", "custom_id"))
            if line.get("url") != params["endpoint"]:
                errors.append(("mismatched_endpoint", i, f"url {line.get('url')!r} differs from the batch endpoint.", "url"))
            seen.add(line.get("custom_id"))
            models.add((line.get("body") or {}).get("model"))
            lines.append(line)
        if len(models) > 1:
            errors.append(("mixed_models", None, f"One model per input file; found {sorted(map(str, models))}.", "body.model"))
        b = {"id": self._id("batch"), "object": "batch", "endpoint": params["endpoint"],
             "completion_window": params["completion_window"], "input_file_id": params["input_file_id"],
             "created_at": NOW, "status": "validating", "metadata": params.get("metadata"),
             "model": next(iter(models)) if len(models) == 1 else None,
             "request_counts": {"total": len(lines), "completed": 0, "failed": 0},
             "_lines": lines, "_errors": errors, "_done": [], "_polls": 0}
        self.batches[b["id"]] = b
        return httpx2.Response(200, json=self._public(b))

    def _advance(self, b: dict) -> dict:
        if b["status"] == "validating":
            if b["_errors"]:
                b["status"] = "failed"
                b["errors"] = {"object": "list", "data": [
                    {"code": c, "line": ln, "message": msg, "param": p} for c, ln, msg, p in b["_errors"]]}
            else:
                b["status"] = "in_progress"
        elif b["status"] == "in_progress":
            b["_polls"] += 1
            for line in b["_lines"][len(b["_done"]):][: self.lines_per_poll]:
                self.attempts[line["custom_id"]] += 1
                status, body = simulated_model(line["body"], self.attempts[line["custom_id"]], n=self.n)
                b["_done"].append((line, status, body))
            ok = sum(s == 200 for _, s, _ in b["_done"])
            b["request_counts"] = {"total": len(b["_lines"]), "completed": ok, "failed": len(b["_done"]) - ok}
            if len(b["_done"]) == len(b["_lines"]):
                b["status"] = "finalizing"
            elif b["_polls"] >= self.window_polls:           # fake: "24 hours" = window_polls polls
                self._finish(b, "expired")
        elif b["status"] == "finalizing":
            self._finish(b, "completed")
        elif b["status"] == "cancelling":
            self._finish(b, "cancelled")                     # fake: unstarted lines appear in neither file
        return b

    def _finish(self, b: dict, status: str) -> None:
        out, err = [], []
        for line, code, body in b["_done"]:
            rec = {"id": self._id("batch_req"), "custom_id": line["custom_id"],
                   "response": {"status_code": code, "request_id": self._id("req"), "body": body}, "error": None}
            (out if code == 200 else err).append(rec)
        if status == "expired":
            for line in b["_lines"][len(b["_done"]):]:
                err.append({"id": self._id("batch_req"), "custom_id": line["custom_id"], "response": None,
                            "error": {"code": "batch_expired",
                                      "message": "This request could not be executed before the completion window expired."}})
                b["request_counts"]["failed"] += 1
        self.rng.shuffle(out)                                # output order may not match input order
        self.rng.shuffle(err)
        usages = [r["response"]["body"]["usage"] for r in out]
        tin, tout = sum(u["input_tokens"] for u in usages), sum(u["output_tokens"] for u in usages)
        b["usage"] = {"input_tokens": tin, "output_tokens": tout, "total_tokens": tin + tout,
                      "input_tokens_details": {"cached_tokens": sum(u["input_tokens_details"]["cached_tokens"] for u in usages)},
                      "output_tokens_details": {"reasoning_tokens": sum(u["output_tokens_details"]["reasoning_tokens"] for u in usages)}}
        b["output_file_id"], b["error_file_id"] = self._store(out), self._store(err)
        b["status"] = status
        b[f"{status}_at"] = NOW + 3600

    # -- responses ------------------------------------------------------------------------------
    def _headers(self, body: dict) -> dict:
        prompt = (body.get("instructions") or "") + json.dumps(body.get("input"))
        charge = max(body.get("max_output_tokens") or 0, rough_tokens(prompt))   # README 7.8 planning rule
        self.used_requests += 1
        self.used_tokens += charge
        return {"x-ratelimit-limit-requests": str(self.rpm), "x-ratelimit-remaining-requests": str(max(0, self.rpm - self.used_requests)),
                "x-ratelimit-limit-tokens": str(self.tpm), "x-ratelimit-remaining-tokens": str(max(0, self.tpm - self.used_tokens)),
                "x-ratelimit-reset-requests": "120ms", "x-ratelimit-reset-tokens": "6m0s"}

    def _respond(self, body: dict) -> httpx2.Response:
        headers = self._headers(body)
        resolved = body.get("service_tier") or "auto"
        resolved = self.project_tier if resolved == "auto" else resolved
        resolved = "priority" if resolved == "fast" else resolved     # error-codes guide: fast is evaluated as priority
        if self.allowed_tiers is not None and resolved in ("default", "flex", "priority") and resolved not in self.allowed_tiers:
            return httpx2.Response(400, headers=headers, json=api_error(   # message, type and param from the error-codes guide
                "Invalid service_tier argument: The requested service tier is not allowed for this project.",
                "invalid_request_error", None, "service_tier"))
        if body.get("service_tier") == "flex" and self.flex_unavailable > 0:
            self.flex_unavailable -= 1                    # fake: the Flex guide publishes no error.code, so this body is invented
            return httpx2.Response(429, json=api_error("Resource Unavailable", "rate_limit_error"), headers=headers)
        if self.script:
            status, payload, extra = self.script.pop(0)
            return httpx2.Response(status, json=payload, headers={**headers, **extra})
        key = hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()
        self.attempts[key] += 1
        status, payload = simulated_model(body, self.attempts[key], n=self.n)
        if status == 200:
            requested = body.get("service_tier") or "auto"
            if requested == "auto":
                requested = self.project_tier             # omitted/"auto" follows the project setting (which can be Fast)
            served = {"flex": "flex", "fast": "fast", "priority": "priority", "ultrafast": "ultrafast"}.get(requested, "default")
            if served in ("fast", "priority") and self.fast_downgrade:
                served = "default"                           # ramped too fast: served (and billed) at Standard
            payload["service_tier"] = served
        return httpx2.Response(status, json=payload, headers=headers)


def offline_client(backend: FakeOpenAI, **options) -> OpenAI:
    """A real OpenAI client whose HTTP traffic goes to `backend` instead of the network."""
    return OpenAI(api_key="sk-offline-demo", base_url="https://fake-openai.invalid/v1",
                  http_client=httpx2.Client(transport=httpx2.MockTransport(backend.handle)), **options)


print("FakeOpenAI ready: offline Files, Batches and Responses behind the real SDK")

## 7.1 The cost levers

Cost per request is uncached input, cached reads, cache writes and output, each at its own price, then multiplied by the processing tier (Batch and Flex 0.5×, Standard 1×, Fast 2×) and, for data residency, by 1.1. **Output includes hidden reasoning tokens**, and output costs 5× input on every GPT-6 model. Read the numbers from a real `ResponseUsage` object, never from a hand-written dict.

[Theory](README.md#71-the-cost-levers)

In [ ]:
# USD per 1M tokens, Standard, short context (<= 272K input tokens).
# From https://developers.openai.com/api/docs/pricing on 2026-10-03. Prices change: keep them in config, with a date.
PRICES = {
    "gpt-6-luna":  {"input": 0.10,  "cached": 0.01, "write": 0.125, "output": 0.50},
    "gpt-6.1-sol": {"input": 2.00,  "cached": 0.10, "write": 2.50,  "output": 10.00},   # cached read is 0.05x on Sol
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "write": 12.50, "output": 50.00},
}
TIER_MULTIPLIER = {"default": 1.0, "auto": 1.0, "batch": 0.5, "flex": 0.5, "fast": 2.0, "priority": 2.0, "ultrafast": 6.0}
ULTRAFAST_MODELS = {"gpt-6-astra", "gpt-6.1-sol"}   # Ultrafast guide; 6x Standard on both
LONG_CONTEXT = 272_000           # above this many input tokens: 2x input, 1.5x output


def usage_of(input_tokens: int, output_tokens: int, *, cached: int = 0, written: int = 0, reasoning: int = 0) -> ResponseUsage:
    """A real SDK usage object (the same type response.usage has)."""
    return ResponseUsage.model_validate({
        "input_tokens": input_tokens, "output_tokens": output_tokens, "total_tokens": input_tokens + output_tokens,
        "input_tokens_details": {"cached_tokens": cached, "cache_write_tokens": written},
        "output_tokens_details": {"reasoning_tokens": reasoning}})


def request_cost(model: str, usage: ResponseUsage, tier: str = "default", *, regional: bool = False) -> float:
    """Dollars for one request. tier = the tier that SERVED it (response.service_tier), or "batch"."""
    if tier == "ultrafast" and model not in ULTRAFAST_MODELS:
        raise ValueError("Ultrafast is offered for gpt-6-astra and gpt-6.1-sol only")
    p, d = PRICES[model], usage.input_tokens_details
    plain = usage.input_tokens - d.cached_tokens - d.cache_write_tokens
    in_mult, out_mult = (2.0, 1.5) if usage.input_tokens > LONG_CONTEXT else (1.0, 1.0)
    dollars = ((plain * p["input"] + d.cached_tokens * p["cached"] + d.cache_write_tokens * p["write"]) * in_mult
               + usage.output_tokens * p["output"] * out_mult) / 1_000_000
    return dollars * TIER_MULTIPLIER[tier] * (1.1 if regional else 1.0)


# The README gotcha: 300 input tokens, 2,000 reasoning tokens, 100 visible output tokens
u = usage_of(300, 2_100, reasoning=2_000)
total = request_cost("gpt-6.1-sol", u)
input_share = 300 * PRICES["gpt-6.1-sol"]["input"] / 1e6 / total
print(f"gpt-6.1-sol: ${total:.6f} per request; input is {input_share:.1%} of it, reasoning is "
      f"{2_000 * PRICES['gpt-6.1-sol']['output'] / 1e6 / total:.1%}")

In [ ]:
# Same request, every lever: model x tier x region
print(f"{'model':<12} " + " ".join(f"{t:>10}" for t in ["batch", "default", "fast", "regional"]))
for model in PRICES:
    row = [request_cost(model, u, t) for t in ["batch", "default", "fast"]] + [request_cost(model, u, regional=True)]
    print(f"{model:<12} " + " ".join(f"${c:.5f}".rjust(10) for c in row))
print("ultrafast, gpt-6-astra:", f"${request_cost('gpt-6-astra', u, 'ultrafast'):.5f}",
      "| gpt-6.1-sol:", f"${request_cost('gpt-6.1-sol', u, 'ultrafast'):.5f}")

> **Observe:** the model column moves cost by 100× (Luna → Astra), the tier by 2× each way, and the region by 10%. For this request, about 3% of the cost is the prompt and over 90% is reasoning: shortening the prompt would barely help, lowering `reasoning.effort` would.
>
> **Try it:** compute the cost of a 300,000-token input on `gpt-6.1-sol` and confirm the long-context multipliers kick in.

## 7.2 Batch API: the lifecycle

Build a JSONL file (one request per line with `custom_id`, `method`, `url`, `body`), upload it with `purpose="batch"`, create the batch for `/v1/responses` with `completion_window="24h"`, wait (poll or webhook), then download the output and error files. 50% off, a separate rate-limit pool, completion within 24 hours.

[Theory](README.md#72-batch-api-the-lifecycle)

In [ ]:
INSTRUCTIONS = (
    "# Identity\nYou extract invoice fields for accounts payable.\n\n"
    "# Instructions\n* Dates ISO 8601, currency ISO 4217. Use null for any value not in the document."
)
INVOICE_SCHEMA = {   # every field required AND nullable, so "use null when missing" is satisfiable (Structured Outputs)
    "type": "object",
    "properties": {
        "vendor": {"type": ["string", "null"]},
        "invoice_date": {"type": ["string", "null"]},
        "currency": {"type": ["string", "null"]},
        "total": {"type": ["number", "null"]},
    },
    "required": ["vendor", "invoice_date", "currency", "total"],
    "additionalProperties": False,
}


def batch_line(custom_id: str, document: str, *, model: str = MODEL, instructions: str = INSTRUCTIONS,
               max_output_tokens: int = 2000) -> dict:
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": model,                                      # ONE model per input file
            "instructions": instructions,                        # identical on every line: cacheable prefix
            "input": f"<document>\n{document}\n</document>",     # the per-request part goes last
            "reasoning": {"effort": "low"},
            "max_output_tokens": max_output_tokens,
            "text": {"format": {"type": "json_schema", "name": "invoice", "schema": INVOICE_SCHEMA, "strict": True}},
        },
    }


# A batch line body is raw JSON, so check it against the SDK's own parameter type before you upload 50,000 of them.
from openai.types.responses.response_create_params import ResponseCreateParamsNonStreaming
BODY_KEYS = ResponseCreateParamsNonStreaming.__required_keys__ | ResponseCreateParamsNonStreaming.__optional_keys__
BODY_ADAPTER = TypeAdapter(ResponseCreateParamsNonStreaming)


def lint_line(line: dict, endpoint: str = "/v1/responses") -> list[str]:
    problems = []
    if set(line) != {"custom_id", "method", "url", "body"}:
        problems.append(f"top-level keys must be custom_id/method/url/body, got {sorted(line)}")
    if line.get("method") != "POST" or line.get("url") != endpoint:
        problems.append(f"method/url must be POST {endpoint}")
    body = line.get("body", {})
    if unknown := set(body) - BODY_KEYS:
        problems.append(f"not Responses API parameters: {sorted(unknown)}")
    if body.get("stream"):
        problems.append("stream=true is not allowed in a batch")
    try:
        BODY_ADAPTER.validate_python(body)
    except ValidationError as err:
        problems.append(f"type error: {err.errors()[0]['loc']} {err.errors()[0]['msg']}")
    return problems


good = batch_line("inv-000123__invoice-v3", "Vendor: Acme Ltd | Date: 2026-03-03 | Currency: EUR | Total: 120.00")
bad = {"custom_id": "1", "method": "POST", "url": "/v1/chat/completions",
       "body": {"model": MODEL, "messages": [{"role": "user", "content": "hi"}], "max_tokens": 50}}
print("good:", lint_line(good) or "OK")
print("bad: ", lint_line(bad))

In [ ]:
TERMINAL = {"completed", "failed", "expired", "cancelled"}


def write_jsonl(path: Path, lines: list[dict]) -> Path:
    path.write_text("".join(json.dumps(line) + "\n" for line in lines), encoding="utf-8")
    return path


def read_jsonl(api: OpenAI, file_id: str | None) -> list[dict]:
    if not file_id:
        return []
    return [json.loads(line) for line in api.files.content(file_id).text.splitlines() if line.strip()]


def submit_batch(api: OpenAI, lines: list[dict], *, metadata: dict):
    """Upload + create. Returns as soon as the batch exists, so the caller can persist its IDs BEFORE polling."""
    path = write_jsonl(WORKDIR / f"{metadata['job']}.jsonl", lines)
    with open(path, "rb") as f:
        input_file = api.files.create(file=f, purpose="batch")
    return api.batches.create(input_file_id=input_file.id, endpoint="/v1/responses",
                              completion_window="24h", metadata=metadata)


def run_batch(api: OpenAI, lines: list[dict], *, metadata: dict, poll_seconds: float = 0, verbose: bool = True,
              on_created=None):
    """Upload -> create -> (on_created: save the IDs) -> poll -> download.
    Live, use poll_seconds=60..300 (or a webhook instead of polling)."""
    batch = submit_batch(api, lines, metadata=metadata)
    if on_created:
        on_created(batch)                    # e.g. write batch_id + input_file_id to the manifest NOW: a crash while
    if verbose:                              # polling must not lose the only handle to a running, billable batch
        print(f"created {batch.id}: {batch.status}")
    while (batch := api.batches.retrieve(batch.id)).status not in TERMINAL:
        if verbose:
            rc = batch.request_counts
            print(f"  {batch.status:<11} completed={rc.completed} failed={rc.failed} total={rc.total}")
        # time.sleep only between polls; live, prefer the batch.completed/failed/expired/cancelled webhooks
        time.sleep(poll_seconds)
    if verbose:
        print(f"final: {batch.status} | output={batch.output_file_id} errors={batch.error_file_id}")
    return batch, read_jsonl(api, batch.output_file_id), read_jsonl(api, batch.error_file_id)


backend = FakeOpenAI(lines_per_poll=2)
api = offline_client(backend)
docs = {
    "inv-000123": "Vendor: Acme Ltd | Date: 2026-03-03 | Currency: EUR | Total: 120.00",
    "inv-000124": "Vendor: Nordlicht GmbH | Date: 2026-03-15 | Currency: EUR | Total: 542.40",
    "inv-000125": "Vendor: Blue Fern Design | Date: 2026-02-28 | Currency: GBP | Total: 1200.00",
    "inv-000126": "Vendor: Kato Electronics | Date: 2026-01-09 | Currency: USD | Total: 90.00",
    "inv-000127": "Vendor: Harbor Cleaning Co. | Date: 2026-05-12 | Currency: CAD | Total: 300.00",
}
lines = [batch_line(f"{doc_id}__invoice-v3", text) for doc_id, text in docs.items()]
batch, output_lines, error_lines = run_batch(api, lines, metadata={"job": "demo-7-2", "prompt_version": "invoice-v3"})
print("batch usage:", batch.usage)
print("HTTP calls the SDK made:", backend.log)

In [ ]:
first = output_lines[0]
body = first["response"]["body"]
print("custom_id:", first["custom_id"], "| status_code:", first["response"]["status_code"], "| body.status:", body["status"])
print("output text:", body["output"][0]["content"][0]["text"])
print("input order :", [line["custom_id"][:10] for line in lines])
print("output order:", [line["custom_id"][:10] for line in output_lines])

> **Observe:** the output order is not the input order. That is documented behavior, and the reason `custom_id` is the only join key ([7.3](README.md#73-designing-custom_ids-and-the-input-file)). Notice also that the raw output body has no `.output_text` shortcut: that is an SDK convenience on `Response` objects, so you join the `output_text` parts yourself (see 7.4).

In [ ]:
# One model per input file. The guide states the rule but not how a violation is reported; this FAKE fails the
# whole batch at validation (status "failed", details in batch.errors), which is the contract for invalid files.
mixed = [batch_line("inv-1__v3", docs["inv-000123"]), batch_line("inv-2__v3", docs["inv-000124"], model="gpt-6.1-sol")]
failed, _, _ = run_batch(api, mixed, metadata={"job": "demo-mixed"}, verbose=False)
print(failed.status)
for e in failed.errors.data:
    print(f"  line={e.line} code={e.code} param={e.param}: {e.message}")
print("(the fake's error codes are illustrative; the documented contract is the status and the errors.data fields)")

In [ ]:
# Webhooks instead of polling: verify the signature, then act on batch.completed / failed / expired / cancelled.
# Here we sign a test event locally with a made-up secret so the SDK's verifier has something to check.
SECRET = "whsec_" + base64.b64encode(b"local-demo-secret-not-a-real-one").decode()
hook_client = OpenAI(api_key="sk-offline-demo", webhook_secret=SECRET)


def sign(body: str, webhook_id: str = "wh_demo", ts: int | None = None) -> dict:
    ts = ts or int(time.time())
    key = base64.b64decode(SECRET.removeprefix("whsec_"))
    sig = base64.b64encode(hmac.new(key, f"{webhook_id}.{ts}.{body}".encode(), hashlib.sha256).digest()).decode()
    return {"webhook-id": webhook_id, "webhook-timestamp": str(ts), "webhook-signature": f"v1,{sig}"}


payload = json.dumps({"id": "evt_demo", "object": "event", "created_at": int(time.time()),
                      "type": "batch.completed", "data": {"id": batch.id}})
event = hook_client.webhooks.unwrap(payload, sign(payload))
print(type(event).__name__, event.type, "->", event.data.id)

tampered = payload.replace(batch.id, "batch_attacker")
try:
    hook_client.webhooks.unwrap(tampered, sign(payload))
except Exception as err:                                  # InvalidWebhookSignatureError
    print("rejected:", type(err).__name__)

> **Try it:** set `backend = FakeOpenAI(lines_per_poll=1, window_polls=3)` and re-run the lifecycle cell. Which status does the batch end in, and where do the unfinished lines show up?

## 7.3 Designing `custom_id`s and the input file

Treat `custom_id` as a primary key you design: unique, derived from the business key, versioned with the prompt, deterministic, short ASCII, no PII. Keep a **manifest** next to every batch, and split files by model, line count (50,000) and size (200 MB).

[Theory](README.md#73-designing-custom_ids-and-the-input-file)

In [ ]:
CUSTOM_ID_RE = re.compile(r"^[A-Za-z0-9_-]{1,64}$")     # Claude's documented pattern; portable on both platforms
SEP = "__"                                              # reserved: may not appear INSIDE either part


def make_custom_id(doc_id: str, prompt_version: str) -> str:
    if SEP in doc_id or SEP in prompt_version:
        raise ValueError(f"{doc_id!r} / {prompt_version!r}: {SEP!r} is the reserved separator, so the ID could not be split back")
    cid = f"{doc_id}{SEP}{prompt_version}"
    if not CUSTOM_ID_RE.fullmatch(cid):
        raise ValueError(f"{cid!r}: use [A-Za-z0-9_-], at most 64 characters, no PII")
    return cid


def doc_id_of(custom_id: str) -> str:
    doc_id, sep, _ = custom_id.partition(SEP)   # unambiguous only because make_custom_id rejects SEP inside a part
    if not sep:
        raise ValueError(f"{custom_id!r} has no {SEP!r} separator")
    return doc_id


for doc_id in ["inv-000123", "jane.doe@example.com", "orders/2026/77", "inv-000123:retry", "region__123"]:
    try:
        print("ok      ", make_custom_id(doc_id, "invoice-v3"))
    except ValueError as err:
        print("rejected", err)

In [ ]:
@dataclass
class Manifest:
    job: str
    model: str
    prompt_version: str
    custom_ids: list[str]
    batches: list[dict] = field(default_factory=list)   # every batch for this job, retries included

    def save(self) -> Path:
        path = WORKDIR / f"{self.job}.manifest.json"
        path.write_text(json.dumps(asdict(self), indent=2), encoding="utf-8")
        return path

    def record(self, batch, role: str = "initial") -> None:
        """Call right after batches.create returns (run_batch's on_created hook), BEFORE polling, and save."""
        self.batches.append({"role": role, "batch_id": batch.id, "input_file_id": batch.input_file_id})
        self.save()

    def missing(self, output_lines: list[dict], error_lines: list[dict]) -> set[str]:
        seen = {line["custom_id"] for line in output_lines + error_lines}
        return set(self.custom_ids) - seen


def chunk_lines(lines: list[dict], *, max_lines: int = 50_000, max_bytes: int = 200 * 1024 * 1024) -> list[list[dict]]:
    """Split into files that respect the per-batch limits. Group by model BEFORE calling this."""
    chunks, current, size = [], [], 0
    for line in lines:
        n = len(json.dumps(line).encode()) + 1
        if n > max_bytes:                     # no chunking can fix one oversized line: shrink it (URL instead of base64)
            raise ValueError(f"{line.get('custom_id')}: one line is {n:,} bytes, over the {max_bytes:,}-byte file limit")
        if current and (len(current) >= max_lines or size + n > max_bytes):
            chunks.append(current)
            current, size = [], 0
        current.append(line)
        size += n
    return chunks + ([current] if current else [])


def split_by_model(lines: list[dict]) -> dict[str, list[dict]]:
    groups: dict[str, list[dict]] = {}
    for line in lines:
        groups.setdefault(line["body"]["model"], []).append(line)
    return groups


many = [batch_line(make_custom_id(f"inv-{i:06d}", "invoice-v3"), f"Vendor: V{i} | Date: 2026-01-01 | Currency: USD | Total: {i}.00",
                   model="gpt-6.1-sol" if i % 10 == 0 else MODEL) for i in range(120)]
for model, group in split_by_model(many).items():
    sizes = [len(c) for c in chunk_lines(group, max_lines=50)]      # tiny limit so the demo splits
    print(f"{model:<12} {len(group):>3} lines -> files of {sizes}")
print("byte-limited:", [len(c) for c in chunk_lines(many, max_bytes=60_000)])
try:
    chunk_lines(many, max_bytes=500)                                 # every line is bigger than this "file limit"
except ValueError as err:
    print("rejected:", err)

manifest = Manifest(job="demo-7-2", model=MODEL, prompt_version="invoice-v3", custom_ids=[l["custom_id"] for l in lines])
manifest.record(batch)                       # in a real run: run_batch(..., on_created=manifest.record), before polling
print("manifest:", manifest.save().name, manifest.batches)
print("missing after download:", manifest.missing(output_lines, error_lines) or "none")

> **Observe:** one job that routes 10% of documents to Sol becomes **two** batches, because a file holds one model. `region__123` is rejected because `__` is the separator `doc_id_of` splits on: allowing it inside a part would truncate the document ID (and `("a__b", "c")` and `("a", "b__c")` would collide). A single line bigger than the file limit is an error, not a one-line chunk. The manifest records every batch's IDs as soon as the batch exists (so a crash while polling cannot orphan a billable batch), and its `missing` check is your audit trail: if it is ever non-empty, a document silently fell through.
>
> **Try it:** add `datetime.now().isoformat()` to a `custom_id` and re-run the builder twice. Why can you no longer diff "submitted" against "returned" across runs?

## 7.4 Reading results and handling failures

A finished batch has two files. Classify every line into one of five failure layers before you decide anything: whole-batch validation failure, `batch_expired`, transient `429`/`5xx`, invalid `4xx`, and application-level failures on a `200` (`incomplete`, refusal-like text, schema or business-rule failure). Resubmit **only** the retryable `custom_id`s, rebuilt from the manifest.

[Theory](README.md#74-reading-results-and-handling-failures)

In [ ]:
class Invoice(BaseModel):            # mirrors INVOICE_SCHEMA: every field required, every field nullable
    vendor: str | None
    invoice_date: str | None
    currency: str | None
    total: float | None


ISO_CURRENCY = re.compile(r"^[A-Z]{3}$")
RESUBMIT_AS_IS = {"batch_expired", "http_429", "http_500", "http_502", "http_503"}
# Billing, spend and quota: retrying cannot help (error-codes guide). Reused by every retry helper below (7.6, 7.9).
STOP_CODES = {"insufficient_quota", "credit_balance_exhausted", "organization_spend_limit_exceeded",
              "project_spend_limit_exceeded", "organization_usage_limit_exceeded"}


def is_billing_error(code: str | None, err_type: str | None) -> bool:
    """Billing errors carry a specific error.code; error.type can still be "insufficient_quota"."""
    return code in STOP_CODES or err_type == "insufficient_quota"


def output_text(body: dict) -> str:
    """Join output_text parts. The raw JSON body has no .output_text shortcut."""
    return "".join(part["text"] for item in body.get("output", []) if item.get("type") == "message"
                   for part in item.get("content", []) if part.get("type") == "output_text")


def classify(line: dict) -> tuple[str, Invoice | None]:
    if line.get("error"):
        return line["error"].get("code") or "error", None                  # e.g. "batch_expired"
    response = line.get("response") or {}
    if response.get("status_code") != 200:
        err = (response.get("body") or {}).get("error") or {}
        if is_billing_error(err.get("code"), err.get("type")):              # a 429 is not always a rate limit
            return f"billing:{err.get('code') or err.get('type')}", None
        return f"http_{response.get('status_code')}", None
    body = response["body"]
    if body.get("status") != "completed":
        return f"incomplete:{(body.get('incomplete_details') or {}).get('reason')}", None
    try:
        invoice = Invoice.model_validate_json(output_text(body))
    except ValidationError:
        return "invalid_output", None
    if invoice.vendor is None or invoice.total is None:
        return "rule_failed:not_an_invoice", None                          # nulls are schema-valid; the rule decides
    if invoice.currency is not None and not ISO_CURRENCY.fullmatch(invoice.currency):
        return "rule_failed:currency", None                                 # schema-valid is not the same as correct
    return "ok", invoice


def bucket(kind: str) -> str:
    if kind == "ok":
        return "ok"
    if kind.startswith("billing:"):
        return "stop"                                                       # alert a human; resubmitting cannot help
    if kind in RESUBMIT_AS_IS or kind.startswith("http_5"):
        return "resubmit"
    if kind.startswith("http_4") or kind.startswith("incomplete"):
        return "fix_first"                                                  # change the request, then resubmit
    return "review"                                                         # wrong content: feedback retry or a human


failure_docs = {
    "inv-000201": "Vendor: Acme Ltd | Date: 2026-03-03 | Currency: EUR | Total: 120.00",
    "inv-000202": "Vendor: Nordlicht GmbH | Date: 2026-03-15 | Currency: EUR | Total: 542.40 [[FLAKY]]",
    "inv-000203": "Vendor: Sakura Foods KK | Date: 2026-06-01 | Currency: JPY | Total: 6500 [[SMUDGED]]",
    "inv-000204": "Vendor: Big Scan Inc | Date: 2026-04-04 | Currency: USD | Total: 10.00 " + "terms and conditions " * 700,
    "inv-000205": "Vendor: Verbose LLC | Date: 2026-04-05 | Currency: USD | Total: 77.00 [[RAMBLING]]",
    "inv-000206": "Scanned photo of a cat, no invoice here.",
    "inv-000207": "Vendor: Kato Electronics | Date: 2026-01-09 | Currency: USD | Total: 90.00",
    "inv-000208": "Vendor: Late Arrivals BV | Date: 2026-07-07 | Currency: EUR | Total: 12.00",
}
backend = FakeOpenAI(lines_per_poll=3, window_polls=2)          # 2 polls x 3 lines: the last 2 lines will expire
api = offline_client(backend)
lines_74 = [batch_line(make_custom_id(d, "invoice-v3"), t) for d, t in failure_docs.items()]
manifest_74 = Manifest(job="demo-7-4", model=MODEL, prompt_version="invoice-v3", custom_ids=[l["custom_id"] for l in lines_74])
batch_74, out_74, err_74 = run_batch(api, lines_74, metadata={"job": "demo-7-4"})

results, failures = {}, {}
for line in out_74 + err_74:
    kind, invoice = classify(line)
    if kind == "ok":
        results[line["custom_id"]] = invoice
    else:
        failures[line["custom_id"]] = kind
for cid, kind in sorted(failures.items()):
    print(f"  {cid:<24} {kind:<28} -> {bucket(kind)}")
print("ok:", sorted(results), "| missing:", manifest_74.missing(out_74, err_74) or "none")

# Not every 429 line is transient: read response.body.error before you call it retryable
spend_line = {"id": "batch_req_x", "custom_id": "inv-000999__invoice-v3", "error": None,
              "response": {"status_code": 429, "request_id": "req_x", "body": api_error(
                  "You exceeded your project spend limit.", "insufficient_quota", "project_spend_limit_exceeded")}}
print("spend-limit 429 line:", classify(spend_line)[0], "->", bucket(classify(spend_line)[0]))

In [ ]:
# Resubmit only what is retryable as-is, rebuilt from the manifest + source docs (never from the output file)
retry_ids = sorted(cid for cid, kind in failures.items() if bucket(kind) == "resubmit")
fix_first = {cid: kind for cid, kind in failures.items() if bucket(kind) == "fix_first"}
review = {cid: kind for cid, kind in failures.items() if bucket(kind) == "review"}
retry_lines = [batch_line(cid, failure_docs[doc_id_of(cid)]) for cid in retry_ids]   # same custom_ids
print("resubmit:", retry_ids)
print("fix first:", fix_first)
print("review:", review)

batch_r, out_r, err_r = run_batch(api, retry_lines, metadata={"job": "demo-7-4-retry"}, verbose=False)
for line in out_r + err_r:
    kind, invoice = classify(line)
    print(f"  retry {line['custom_id']:<24} -> {kind}")

In [ ]:
# Cancelling: a prompt bug spotted early. cancelling -> cancelled; unfinished lines need a fresh batch (new version suffix).
backend = FakeOpenAI(lines_per_poll=2)
api = offline_client(backend)
path = write_jsonl(WORKDIR / "cancel-demo.jsonl", lines_74)
with open(path, "rb") as f:
    cancel_me = api.batches.create(input_file_id=api.files.create(file=f, purpose="batch").id,
                                   endpoint="/v1/responses", completion_window="24h")
api.batches.retrieve(cancel_me.id)                       # validating -> in_progress
api.batches.retrieve(cancel_me.id)                       # 2 lines done
print(api.batches.cancel(cancel_me.id).status)
cancelled = api.batches.retrieve(cancel_me.id)
print(cancelled.status, cancelled.request_counts)

> **Observe:** the `[[FLAKY]]` line (a `500`) and the two expired lines succeed when resubmitted unchanged. The oversized document (`http_400`) and the truncated one (`incomplete:max_output_tokens`) would fail the same way again, so they go to **fix first** (shorten or split the input; raise the output limit). The smudged currency and the cat photo (all fields `null`: schema-valid, but not an invoice) are `200`s with wrong content: those need a retry with feedback (chapter 06) or a human, not a resubmission. And a `429` line whose body carries a spend or quota code goes to **stop**: resubmitting it would fail the same way until a human fixes billing.
>
> **Try it:** re-run the resubmission cell with *all* failures instead of `retry_ids`. Count how many lines come back failed again, and what that would cost on a 100,000-document job.

## 7.5 SLA planning and choosing sync, background, Flex or Batch

Plan with the 24-hour worst case: `latest_submit = deadline − 24 h − retry_path − buffer`. A second batch as the retry path costs another 24 hours. Then pick the processing option from two questions: is someone waiting, and what is the deadline?

[Theory](README.md#75-sla-planning-and-choosing-sync-background-flex-or-batch)

In [ ]:
def latest_submit(deadline: datetime, *, retry_path_h: float = 0, buffer_h: float = 0, window_h: float = 24) -> datetime:
    return deadline - timedelta(hours=window_h + retry_path_h + buffer_h)


now = datetime(2026, 10, 3, 9, 0, tzinfo=timezone.utc)
deadline = now + timedelta(hours=30)
for label, retry_h, buffer_h in [("no retry path, no buffer", 0, 0),
                                 ("retry via Flex/Standard (3 h) + 1 h buffer", 3, 1),
                                 ("retry via a second batch (24 h)", 24, 0)]:
    t = latest_submit(deadline, retry_path_h=retry_h, buffer_h=buffer_h)
    verdict = f"submit by {t:%a %H:%M} UTC ({(t - now).total_seconds() / 3600:+.0f} h from now)"
    print(f"{label:<44} {verdict}{'  -> IMPOSSIBLE' if t < now else ''}")

In [ ]:
def choose_processing(*, user_waiting: bool, latency_is_product: bool = False, deadline_h: float | None = None,
                      requests: int = 1, long_running: bool = False) -> str:
    """A first-cut chooser that mirrors the README 7.5 comparison table. Real choices also weigh evals and limits."""
    if user_waiting:
        return "fast" if latency_is_product else "standard (sync/streaming)"
    if long_running and (deadline_h is None or deadline_h < 1):
        return "background=True (Standard price)"
    if deadline_h is not None and deadline_h <= 24:
        return "flex (+ backoff, Standard fallback)"
    return "batch" if requests >= 100 else "flex (+ backoff, Standard fallback)"


SCENARIOS = [
    ("Pre-merge PR check, developer waiting", dict(user_waiting=True)),
    ("Checkout fraud score, latency is the product", dict(user_waiting=True, latency_is_product=True)),
    ("Single 8-minute reasoning job for a dashboard", dict(user_waiting=False, long_running=True)),
    ("Enrich 5,000 leads within 4 hours", dict(user_waiting=False, deadline_h=4, requests=5_000)),
    ("Re-score 400,000 tickets by next week", dict(user_waiting=False, deadline_h=24 * 7, requests=400_000)),
    ("Nightly report, results by 07:00 (submitted 30 h ahead)", dict(user_waiting=False, deadline_h=30, requests=20_000)),
]
for name, kwargs in SCENARIOS:
    print(f"{name:<56} -> {choose_processing(**kwargs)}")

> **Observe:** the 30-hour case only works with no retry path or a fast one; a second batch makes it impossible. `background=True` appears only for long single jobs, and it is never a discount.
>
> **Try it:** add a rule for a 2-hour deadline with 50,000 requests. Is Flex realistic at your rate limits (7.8), or do you need to negotiate the deadline?

## 7.6 Service tiers: `service_tier`

`service_tier` picks the processing tier per request, and `response.service_tier` reports the tier that **actually served** it. Flex gives batch prices on a synchronous call but can return `429` (not charged): back off, then fall back to Standard when the deadline needs it. Fast mode can be downgraded to Standard when traffic ramps too fast, so group your cost dashboard by the served tier.

[Theory](README.md#76-service-tiers-service_tier)

In [ ]:
SERVICE_TIERS = typing.get_args(typing.get_args(Response.model_fields["service_tier"].annotation)[0])
print("service_tier values in openai", __import__("openai").__version__, "->", SERVICE_TIERS)

def server_delay(headers) -> float | None:
    """Seconds the server asked us to wait, or None. Retry-After is documented in seconds; it may also be an HTTP date
    (RFC 9110), and the SDK additionally reads retry-after-ms. Treat the value as a minimum, never as a suggestion."""
    for name, scale in (("retry-after-ms", 0.001), ("retry-after", 1.0)):
        value = headers.get(name)
        if value is None:
            continue
        try:
            seconds = float(value) * scale
        except ValueError:
            continue                                          # maybe an HTTP date: handled below
        if not math.isnan(seconds):
            return max(0.0, seconds)
    value = headers.get("retry-after")
    try:
        when = email.utils.parsedate_to_datetime(value) if value else None
        return None if when is None else max(0.0, (when - datetime.now(timezone.utc)).total_seconds())
    except (TypeError, ValueError):
        return None


def create_flex_with_fallback(api: OpenAI, *, deadline_s: float, standard_reserve_s: float = 0.0,
                              base_delay: float = 1.0, sleep=time.sleep, **kwargs) -> Response:
    """Flex first; Standard when Flex cannot finish in time. deadline_s bounds the WHOLE call: Flex may use the time up to
    (deadline - standard_reserve_s), the reserve is kept for the Standard request, and nothing starts after the deadline."""
    deadline = time.monotonic() + deadline_s
    for attempt in range(6):
        flex_left = deadline - standard_reserve_s - time.monotonic()
        if flex_left <= 0:
            break
        try:
            flex = api.with_options(timeout=min(900.0, flex_left), max_retries=0)   # we own the loop; never outlive the budget
            return flex.responses.create(service_tier="flex", **kwargs)
        except APITimeoutError:
            break                                             # Flex too slow for this deadline: use the reserve on Standard
        except RateLimitError as err:
            if is_billing_error(err.code, err.type):
                raise                                         # billing and quota: retrying cannot help
            delay = max(server_delay(err.response.headers) or 0.0, min(60.0, base_delay * 2 ** attempt))
            delay += random.uniform(0, base_delay)            # jitter on top of the server's minimum
            if time.monotonic() + delay > deadline - standard_reserve_s:
                break                                         # the required wait does not fit: fall back now
            sleep(delay)
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError("deadline reached before the Standard fallback could start; defer the request")
    standard = api.with_options(timeout=remaining, max_retries=0)
    # service_tier="default" on purpose: omitting it (or "auto") follows the PROJECT tier, which may be set to Fast (2x)
    return standard.responses.create(service_tier="default", **kwargs)


request = dict(model=MODEL, instructions=INSTRUCTIONS, input=f"<document>\n{docs['inv-000123']}\n</document>")
UNAVAILABLE = api_error("Resource Unavailable", "rate_limit_error")   # fake body: the Flex guide publishes no error.code

# deadline_s, standard_reserve_s: the close-deadline case keeps 0.3 s for Standard, so the fallback always has time
for label, unavailable, script, deadline_s, reserve_s, project_tier in [
        ("capacity returns after 2 tries", 2, [], 60, 0.0, "default"),
        ("no capacity, deadline is close", 10**6, [], 0.5, 0.3, "default"),
        ("429 asks for 120 s, 60 s budget", 0, [(429, UNAVAILABLE, {"retry-after": "120"})], 60, 0.0, "default"),
        ("same, project tier set to Fast", 0, [(429, UNAVAILABLE, {"retry-after": "120"})], 60, 0.0, "fast")]:
    backend = FakeOpenAI(project_tier=project_tier)
    backend.flex_unavailable = unavailable              # the next N Flex requests get "429 Resource Unavailable"
    backend.script = list(script)
    waits = []
    r = create_flex_with_fallback(offline_client(backend), deadline_s=deadline_s, standard_reserve_s=reserve_s, base_delay=0.01,
                                  sleep=lambda d: (waits.append(d), time.sleep(d)), **request)   # really sleeps (ms)
    print(f"{label:<33} served by: {r.service_tier:<8} HTTP calls: {len(backend.log)}  slept: {sum(waits):5.2f} s  "
          f"cost: ${request_cost(MODEL, r.usage, r.service_tier):.8f}")

backend = FakeOpenAI()
backend.script = [(429, api_error("You exceeded your project spend limit.", "insufficient_quota", "project_spend_limit_exceeded"), {})]
try:
    create_flex_with_fallback(offline_client(backend), deadline_s=60, base_delay=0.01, **request)
except RateLimitError as err:
    print("stopped immediately on", err.code, "after", len(backend.log), "call")
backend = FakeOpenAI()
try:
    create_flex_with_fallback(offline_client(backend), deadline_s=0, **request)
except TimeoutError as err:
    print(f"deadline 0 s: {err} ({len(backend.log)} HTTP calls)")
in_90_s = email.utils.format_datetime(datetime.now(timezone.utc) + timedelta(seconds=90), usegmt=True)
print("server_delay:", server_delay({"retry-after-ms": "1500"}), "|", server_delay({"retry-after": "2"}), "|",
      round(server_delay({"retry-after": in_90_s})), f"(from {in_90_s!r})")

In [ ]:
# Fast mode ramped too quickly: some requests come back served (and billed) at Standard
backend = FakeOpenAI(fast_downgrade=True)
api = offline_client(backend)
by_tier = Counter()
for i in range(3):
    r = api.responses.create(service_tier="fast", **request)
    by_tier[(r.service_tier, "requested fast")] += request_cost(MODEL, r.usage, r.service_tier)
print({k: f"${v:.8f}" for k, v in by_tier.items()})

> **Observe:** with capacity returning, Flex served the request at half price after three HTTP calls. When the deadline was too close, the wrapper paid Standard rather than miss it. When the `429` asked for 120 seconds and only 60 were left, the wrapper did not retry sooner than the server allowed; it fell back at once. The fallback sends `service_tier="default"` explicitly, so it is served at Standard even when the project tier is Fast (omitting the parameter, or `"auto"`, would have paid 2×). With a zero budget nothing is dispatched at all: the deadline bounds the whole call, including each request's timeout. A spend-limit `429` stopped at once: backing off cannot fix billing. And the "fast" requests were billed as `default`, which only shows up if you log the **served** tier.
>
> **Try it:** remove `err_type == "insufficient_quota"` from `is_billing_error` (7.4) and send a `429` with `type="insufficient_quota", code=None`. What happens, and why is checking both fields safer?

## 7.7 Prompt caching

On GPT-6 models, prefixes of at least 1,024 tokens are cacheable; writes cost 1.25× input with the default 30-minute lifetime (2.5× with `ttl="12h"`) and reads 0.1× (0.05× on `gpt-6.1-sol`). In implicit mode the breakpoint sits at the end of the latest user message, so a shared prefix followed by a changing document is **written but never read**. Put an explicit `prompt_cache_breakpoint` after the static content (inside a `developer` message, not `instructions`), and consider explicit-only mode.

[Theory](README.md#77-prompt-caching)

In [ ]:
# What the SDK types say (openai 3.24): where a breakpoint can go and which cache options exist.
# The prompt-caching guide also documents ttl="12h" (2.5x write); openai 3.24 still types only "30m".
from openai.types.responses import ResponseInputTextParam
from openai.types.responses.response_create_params import PromptCacheOptions

print("input_text block fields:", list(typing.get_type_hints(ResponseInputTextParam)))
print("prompt_cache_options:  ", {k: v for k, v in typing.get_type_hints(PromptCacheOptions).items()})
sig = typing.get_type_hints(OpenAI(api_key="sk-offline-demo").responses.create)
print("instructions type:     ", sig["instructions"], "(a plain string: no place for a breakpoint)")

In [ ]:
RULES = ("# Identity\nYou extract invoice fields for accounts payable.\n\n# Instructions\n"
         "* Dates ISO 8601, currency ISO 4217. Use null for any value not in the document.\n\n# Examples\n"
         + "\n".join(f'<example id="{i}">Vendor: Example {i} | Date: 2026-0{1 + i % 9}-1{i % 10} | Currency: EUR | '
                     f'Total: {i * 37}.00 -> {{"vendor": "Example {i}", "currency": "EUR", "total": {i * 37}.0}}</example>'
                     for i in range(40)))
print("static prefix ~", rough_tokens(RULES), "tokens (needs >= 1,024 to be cacheable)")


class ToyPromptCache:
    """A toy model of the documented GPT-6 rules (NOT OpenAI's implementation): 1,024-token minimum,
    implicit breakpoint at the end of the latest user message, explicit breakpoints where marked, up to 4 writes
    per request. Lookups use only this request's breakpoints; the real lookup set also includes earlier message
    endings, which matters for multi-turn chats, not for this single-turn extraction."""
    MIN = 1024

    def __init__(self):
        self.store: set[str] = set()

    def request(self, blocks: list[tuple[str, bool]], mode: str = "implicit") -> ResponseUsage:
        """blocks: (text, has_explicit_breakpoint) in prompt order."""
        breakpoints, pos, h = [], 0, hashlib.sha256()
        for text, explicit in blocks:
            pos += rough_tokens(text)
            h.update(text.encode())
            if explicit:
                breakpoints.append((pos, h.hexdigest()))
        if mode == "implicit" and (not breakpoints or breakpoints[-1][0] != pos):
            breakpoints.append((pos, h.hexdigest()))
        eligible = [(p, k) for p, k in breakpoints if p >= self.MIN]
        cached = max((p for p, k in eligible if k in self.store), default=0)
        new = [(p, k) for p, k in eligible if p > cached and k not in self.store][:4]
        self.store.update(k for _, k in new)
        written = new[-1][0] - cached if new else 0
        return usage_of(pos, 60, cached=cached, written=written)


DOCS = [f"Vendor: Customer {i} | Date: 2026-09-0{i} | Currency: USD | Total: {i * 11}.00 " + "line item " * 40 for i in range(1, 6)]
PATTERNS = {
    "implicit, rules + doc":                lambda d: ([(RULES, False), (d, False)], "implicit"),
    "implicit, timestamp before the rules": lambda d: ([(f"Now: {time.time_ns()}\n" + RULES, False), (d, False)], "implicit"),
    "explicit bp after rules, implicit on": lambda d: ([(RULES, True), (d, False)], "implicit"),
    "explicit-only, bp after rules":        lambda d: ([(RULES, True), (d, False)], "explicit"),
}
for model in ["gpt-6-luna", "gpt-6.1-sol"]:
    uncached = sum(request_cost(model, usage_of(rough_tokens(RULES + d), 60)) for d in DOCS)
    print(f"\n{model}: 5 requests without any caching = ${uncached:.6f}")
    for name, build in PATTERNS.items():
        cache, cost, reads, writes = ToyPromptCache(), 0.0, 0, 0
        for d in DOCS:
            usage = cache.request(*build(d))
            cost += request_cost(model, usage)
            reads += usage.input_tokens_details.cached_tokens
            writes += usage.input_tokens_details.cache_write_tokens
        print(f"  {name:<38} cached={reads:>5} written={writes:>5}  ${cost:.6f} ({cost / uncached:.0%} of uncached)")

In [ ]:
def break_even_length(M: int = 1024, r: float = 0.1, w: float = 1.25, N: int = 10) -> float:
    """Shortest original prefix worth padding to the minimum M (OpenAI caching guide): M * (r + (w - r) / N)."""
    return M * (r + (w - r) / N)


for n in [2, 10, 100, 10_000]:
    L = break_even_length(N=n)
    print(f"N={n:>6}: pad a short shared prefix to 1,024 tokens if it is longer than {L:6.1f} tokens "
          f"(smallest whole length: {int(L) + 1})")
print("one write + 1 reuse =", 1.25 + 0.1, "x | one write + 9 reuses =", 1.25 + 9 * 0.1, "x (instead of 2x and 10x)")

In [ ]:
# The request shape that caches the rules but never writes the document (explicit-only mode)
def cached_extraction_request(document: str, tenant: str) -> dict:
    return dict(
        model=MODEL,
        prompt_cache_options={"mode": "explicit"},
        prompt_cache_key=f"invoices:{tenant}",                    # per-tenant cache accounting
        input=[
            {"role": "developer", "content": [{"type": "input_text", "text": RULES,
                                               "prompt_cache_breakpoint": {"mode": "explicit"}}]},
            {"role": "user", "content": f"<document>\n{document}\n</document>"},
        ],
        reasoning={"effort": "low"},
        max_output_tokens=2000,
    )


req = cached_extraction_request(DOCS[0], "tenant-042")
print(json.dumps({**req, "input": [{**req["input"][0], "content": [{**req["input"][0]["content"][0], "text": "<RULES...>"}]},
                                   req["input"][1]]}, indent=1)[:900])

In [ ]:
# (live) Two requests sharing the explicit prefix. Expect cache_write_tokens on the first, cached_tokens on the second.
for i, doc in enumerate(DOCS[:2], start=1):
    r = client.responses.create(**cached_extraction_request(doc, "tenant-042"))
    d = r.usage.input_tokens_details
    print(f"request {i}: input={r.usage.input_tokens} cached={d.cached_tokens} written={d.cache_write_tokens} "
          f"cost=${request_cost(MODEL, r.usage, r.service_tier or 'default'):.6f}")

> **Observe:** in implicit mode every request **writes** the rules plus its own document and nothing is read, so caching costs *more* than no caching (the 1.25× write is pure overhead). An explicit breakpoint after the rules turns later requests into 0.1× reads; explicit-only mode also stops paying to write each document. The timestamp pattern never hits at all. The toy cache ignores details such as hidden system tokens, but the arithmetic follows the documented multipliers.
>
> **Try it:** cut `RULES` to 25 examples so the prefix drops under 1,024 tokens, and re-run the toy comparison. Then use `break_even_length` to decide whether padding it back is worth it at 10 requests.

## 7.8 Rate limits, usage tiers and spend controls

Limits are per organization, project and model, in RPM and TPM; whichever you hit first applies. For planning, the rate-limits guide says a request's token charge is the larger of `max_tokens` and the estimated token count of the request. The guide words this with `max_tokens`; we assume `max_output_tokens` plays that role on the Responses API, and OpenAI does not publish the estimator, so treat the numbers below as planning approximations. Either way, an oversized cap eats TPM you never use. Cached tokens still count toward TPM. Read the `x-ratelimit-*` headers with `with_raw_response`, and keep spend limits separate from rate limits.

[Theory](README.md#78-rate-limits-usage-tiers-and-spend-controls)

In [ ]:
# gpt-6-luna default limits from its model page (2026-11-06); your organization's limits page is authoritative
LUNA_TIERS = {  # paid tier: (RPM, TPM, batch queue tokens)
    "Build": (5_000, 2_000_000, 20_000_000),
    "Launch": (10_000, 10_000_000, 1_000_000_000),
    "Grow": (30_000, 180_000_000, 15_000_000_000),
}


def tpm_charge(prompt_tokens: int, max_output_tokens: int) -> int:
    """Planning rule from the rate-limits guide: the larger of max_tokens and the estimated request size.
    Assumption: max_output_tokens plays the max_tokens role on Responses. The estimator is not published,
    so treat this as a planning approximation, not the exact charge."""
    return max(max_output_tokens, prompt_tokens)


def max_requests_per_minute(tier: str, prompt_tokens: int, max_output_tokens: int) -> int:
    rpm, tpm, _ = LUNA_TIERS[tier]
    return min(rpm, tpm // tpm_charge(prompt_tokens, max_output_tokens))


for cap in [32_000, 8_000, 2_000]:
    print(f"max_output_tokens={cap:>6}: Build -> {max_requests_per_minute('Build', 1_500, cap):>4} req/min, "
          f"Launch -> {max_requests_per_minute('Launch', 1_500, cap):>5} req/min")


def smallest_tier(rpm_needed: int, tpm_needed: int, queued_batch_tokens: int = 0) -> str | None:
    for tier, (rpm, tpm, queue) in LUNA_TIERS.items():
        if rpm >= rpm_needed and tpm >= tpm_needed and queue >= queued_batch_tokens:
            return tier
    return None


print("smallest tier for 2,000 req/min x 2,000 tokens:", smallest_tier(2_000, 4_000_000))
print("smallest tier to queue a 100,000-line batch of 2,000-token prompts:", smallest_tier(0, 0, 200_000_000))

In [ ]:
def reset_seconds(value: str) -> float:
    """'6m0s' -> 360.0, '1s' -> 1.0, '120ms' -> 0.12"""
    units = {"h": 3600, "m": 60, "s": 1, "ms": 0.001}
    return sum(float(n) * units[u] for n, u in re.findall(r"([\d.]+)(ms|h|m|s)", value))


backend = FakeOpenAI(rpm=5_000, tpm=2_000_000)   # Build-tier Luna limits
api = offline_client(backend)
for cap in [2_000, 32_000]:
    raw = api.responses.with_raw_response.create(**request, max_output_tokens=cap)
    h = raw.headers
    print(f"cap={cap:>6}: remaining requests {h['x-ratelimit-remaining-requests']}/{h['x-ratelimit-limit-requests']}, "
          f"remaining tokens {h['x-ratelimit-remaining-tokens']}/{h['x-ratelimit-limit-tokens']}, "
          f"tokens reset in {reset_seconds(h['x-ratelimit-reset-tokens'])} s")
response = raw.parse()                                     # the usual Response object
print("parsed:", response.status, response.usage.output_tokens, "output tokens actually used")

In [ ]:
SPEND_CONTROLS = {   # error.code on the 429 -> who acts
    "organization_spend_limit_exceeded": "org owner raises the hard limit (or wait for the monthly reset)",
    "project_spend_limit_exceeded": "project owner raises the project limit; other projects keep working",
    "organization_usage_limit_exceeded": "request a higher OpenAI usage limit (tier-based)",
    "credit_balance_exhausted": "add prepaid credits",
}
for code_, action in SPEND_CONTROLS.items():
    print(f"{code_:<36} -> stop and alert: {action}")

> **Observe:** on the Build tier (2,000,000 TPM) a 32,000-token cap limits you to about 62 requests per minute even though each request uses a few hundred output tokens; the same TPM supports 1,000 per minute with a 2,000-token cap. Size caps per task from observed `usage`. The fake server's headers show the same effect on `x-ratelimit-remaining-tokens`.
>
> **Try it:** a nightly job needs 100,000 requests of 2,000 tokens in 8 hours. Compute the required RPM and TPM, then compare `smallest_tier` for the synchronous path against the batch queue path.

## 7.9 Retries and backoff

Classify by HTTP status **and** `error.code`: back off on rate limits and overload, reduce the rate on `slow_down`, stop on billing and quota, fix the request on `400`. The SDK already retries `408`, `409`, `429`, `5xx` and connection errors twice; if you own the retry loop, set `max_retries=0` on that client so retries don't multiply.

[Theory](README.md#79-retries-and-backoff)

In [ ]:
def retry_action(status: int | None, code: str | None = None, *, flex: bool = False, err_type: str | None = None) -> str:
    """status=None means a connection error or timeout."""
    if is_billing_error(code, err_type):
        return "stop_and_alert"
    if status is None or status in (408, 409) or status >= 500:
        return "backoff"                         # 503 server_is_overloaded included: honor Retry-After first
    if status == 429:
        if code == "slow_down":
            return "reduce_rate"
        return "flex_backoff_or_fallback" if flex else "backoff"
    if status in (401, 403):
        return "stop_and_alert"                  # auth, region, or misalignment_policy_violation: no auto-retry
    return "fix_request"                         # 400 and other 4xx: the same bytes fail the same way


CASES = [(429, "rate_limit_exceeded", False), (429, "slow_down", False), (429, None, True),
         (429, "project_spend_limit_exceeded", True), (503, "server_is_overloaded", False), (500, None, False),
         (None, None, False), (400, "context_length_exceeded", False), (403, "misalignment_policy_violation", False)]
for status, code_, flex in CASES:
    print(f"{status!s:>4} {code_ or '-':<32} flex={flex!s:<5} -> {retry_action(status, code_, flex=flex)}")
print("(rate_limit_exceeded and context_length_exceeded are illustrative codes; check the bodies you actually receive)")

In [ ]:
# What the SDK does on its own, measured against the fake server (503 with a tiny Retry-After)
OVERLOADED = (503, api_error("The model is overloaded.", "service_unavailable_error", "server_is_overloaded"), {"retry-after": "0.01"})

for label, options in [("SDK default (max_retries=2)", {}), ("max_retries=0", {"max_retries": 0})]:
    backend = FakeOpenAI()
    backend.script = [OVERLOADED] * 10
    try:
        offline_client(backend, **options).responses.create(**request)
    except InternalServerError as err:
        print(f"{label:<28} -> {len(backend.log)} HTTP request(s), then {type(err).__name__} ({err.code})")

# Nested retries: your 3-attempt loop around an SDK that also retries twice
backend = FakeOpenAI()
backend.script = [OVERLOADED] * 50
nested = offline_client(backend)
for attempt in range(3):
    try:
        nested.responses.create(**request)
    except InternalServerError:
        pass
print("nested loop: 3 application attempts ->", len(backend.log), "HTTP requests during the incident")

In [ ]:
def call_with_backoff(api: OpenAI, *, max_attempts: int = 5, max_total_s: float = 120.0, base_delay: float = 1.0,
                      sleep=time.sleep, **kwargs) -> Response:
    """Our own retry loop: SDK retries off, every status classified by retry_action, Retry-After honored as a minimum,
    and max_total_s bounding the whole call (each request's timeout is the time left, not the SDK's 600 s default)."""
    deadline = time.monotonic() + max_total_s
    for attempt in range(max_attempts):
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            break
        try:
            return api.with_options(max_retries=0, timeout=remaining).responses.create(**kwargs)
        except APIStatusError as err:                           # every HTTP error, including 408 and 409 (ConflictError)
            action = retry_action(err.status_code, err.code, err_type=err.type)
            if action not in ("backoff", "reduce_rate"):
                raise                                           # stop_and_alert or fix_request: page a human / fix it
            delay = max(server_delay(err.response.headers) or 0.0, base_delay * 2 ** attempt)
        except APIConnectionError:                              # includes APITimeoutError
            delay = base_delay * 2 ** attempt
        delay += random.uniform(0, base_delay)                  # jitter: don't retry in lockstep
        if attempt == max_attempts - 1 or time.monotonic() + delay > deadline:
            break                                               # the required wait does not fit in the budget
        sleep(delay)
    raise RuntimeError("retry budget exhausted; defer the request (queue it, or move it to Flex/Batch)")


waits = []
backend = FakeOpenAI()
backend.script = [(429, api_error("Slow down.", "rate_limit_error", "slow_down"), {"retry-after": "2"}), OVERLOADED,
                  (408, api_error("Request timed out.", "timeout_error"), {}),
                  (409, api_error("Conflict.", "invalid_request_error"), {})]
r = call_with_backoff(offline_client(backend), base_delay=0.01, sleep=waits.append, **request)
print("succeeded after", len(backend.log), "requests (429 slow_down, 503, 408, 409, then 200); waited",
      [round(w, 2) for w in waits], "s")

for label, headers in [("Retry-After: 120", {"retry-after": "120"}),
                       ("Retry-After as an HTTP date, ~90 s away", {"retry-after": email.utils.format_datetime(
                           datetime.now(timezone.utc) + timedelta(seconds=90), usegmt=True)}),
                       ("retry-after-ms: 90000", {"retry-after-ms": "90000"})]:
    backend, waits = FakeOpenAI(), []
    backend.script = [(429, api_error("Rate limit reached.", "rate_limit_error"), headers)]
    try:
        call_with_backoff(offline_client(backend), max_total_s=60, base_delay=0.01, sleep=waits.append, **request)
    except RuntimeError as err:
        print(f"{label:<40} -> {len(backend.log)} request, slept {sum(waits)} s, then: {err}")

backend = FakeOpenAI()
backend.script = [OVERLOADED] * 10
try:
    call_with_backoff(offline_client(backend), max_attempts=4, base_delay=0.01, sleep=lambda s: None, **request)
except RuntimeError as err:
    print(len(backend.log), "requests, then:", err)

> **Observe:** one failing call with SDK defaults is **3** HTTP requests; wrapped in your own 3-attempt loop it becomes **9**, every one of which also counts against your rate limit during an incident. With `max_retries=0` on the client your loop is the only one. It honors the server's delay as a minimum in all three forms (seconds, an HTTP date, `retry-after-ms`), still retries `408` and `409` (which the SDK would have retried for you), and when the required wait does not fit in its budget it defers instead of retrying early. Also note that openai 3.24 does **not** retry at all when `Retry-After` exceeds 120 seconds (`MAX_RETRY_AFTER_DELAY`): the error reaches your handler at once, so your handler must defer the request rather than retry sooner.
>
> **Try it:** script five `503`s with `Retry-After: 20` and call `call_with_backoff(..., max_total_s=45)` with a recording `sleep`. How many requests go out before it defers, and how long did it wait in total?

## 7.10 Cost modeling per request and per workload

Per request: apply the formula to `response.usage` (or each `body.usage` in a batch output file). Per workload: multiply by volume and add retries, escalations, human review and evals. Report **cost per successful task**, and reconcile daily with the Costs API.

[Theory](README.md#710-cost-modeling-per-request-and-per-workload)

In [ ]:
# README worked example: 2,000 input tokens (1,200 of them a cached prefix in steady state), 400 output tokens
steady = usage_of(2_000, 400, cached=1_200)
cold = usage_of(2_000, 400)
print(f"{'model':<12} {'per request':>12} {'100k Standard':>14} {'100k Batch':>11} {'no cache':>10}")
for model in PRICES:
    per = request_cost(model, steady)
    print(f"{model:<12} ${per:>11.6f} ${per * 100_000:>13,.2f} ${request_cost(model, steady, 'batch') * 100_000:>10,.2f} "
          f"${request_cost(model, cold):>9.6f}")

In [ ]:
def cost_per_successful_task(*, per_request: float, success_rate: float, human_review_cost: float | None = None) -> float:
    """No review: failed tasks are wasted spend, so divide by the success rate.
    With review: a human fixes every failure, so every task ends correct and pays tokens + its share of review."""
    if human_review_cost is None:
        return per_request / success_rate
    return per_request + (1 - success_rate) * human_review_cost


luna, sol = request_cost("gpt-6-luna", steady), request_cost("gpt-6.1-sol", steady)
for name, per, rate in [("Luna", luna, 0.92), ("Sol", sol, 0.99)]:
    print(f"{name:<4} pass={rate:.0%}: tokens only ${cost_per_successful_task(per_request=per, success_rate=rate):.6f} | "
          f"with $0.50 human review of failures ${cost_per_successful_task(per_request=per, success_rate=rate, human_review_cost=0.50):.4f}"
          " per successful document")

In [ ]:
def cost_log_record(response: Response, *, requested_tier: str, prompt_version: str, latency_s: float) -> dict:
    """One row per request: enough to rebuild any cost dashboard later (group by served tier, not requested)."""
    u = response.usage
    return {"model": response.model, "requested_tier": requested_tier, "served_tier": response.service_tier,
            "input": u.input_tokens, "cached": u.input_tokens_details.cached_tokens,
            "written": u.input_tokens_details.cache_write_tokens, "output": u.output_tokens,
            "reasoning": u.output_tokens_details.reasoning_tokens, "prompt_version": prompt_version,
            "latency_s": round(latency_s, 3), "usd": round(request_cost(response.model, u, response.service_tier or "default"), 8)}


api = offline_client(FakeOpenAI())
t0 = time.monotonic()
r = api.responses.create(service_tier="flex", **request)
print(cost_log_record(r, requested_tier="flex", prompt_version="invoice-v3", latency_s=time.monotonic() - t0))

In [ ]:
# (live) Count tokens before a big batch, then project the cost. Count the SAME payload you will send: derive the
# parameters from the real batch line body (text.format schema, reasoning, tools...), dropping only what count() lacks.
from openai.types.responses.input_token_count_params import InputTokenCountParams
COUNT_KEYS = InputTokenCountParams.__required_keys__ | InputTokenCountParams.__optional_keys__


def count_params(body: dict) -> dict:
    return {k: v for k, v in body.items() if k in COUNT_KEYS}


sample = [failure_docs["inv-000201"], failure_docs["inv-000207"], docs["inv-000124"]]
bodies = [batch_line(make_custom_id(f"sample-{i}", "invoice-v3"), d)["body"] for i, d in enumerate(sample)]
print("counted:", sorted(count_params(bodies[0])), "| not accepted by count():", sorted(set(bodies[0]) - COUNT_KEYS))
counts = [client.responses.input_tokens.count(**count_params(b)).input_tokens for b in bodies]
avg = sum(counts) / len(counts)
projected = request_cost(MODEL, usage_of(int(avg), 400), "batch") * 200_000
print(f"avg input tokens {avg:.0f} -> 200,000 documents via Batch ~ ${projected:,.2f} (assuming 400 output tokens each)")

In [ ]:
# (live, optional) Daily reconciliation with the Costs API. Needs an Admin API key (OPENAI_ADMIN_KEY), not a project key.
if not os.getenv("OPENAI_ADMIN_KEY"):
    print("OPENAI_ADMIN_KEY not set; skipping the Costs API call")
else:
    admin = OpenAI(admin_api_key=os.environ["OPENAI_ADMIN_KEY"])
    start = int((datetime.now(timezone.utc) - timedelta(days=7)).timestamp())
    page = admin.admin.organization.usage.costs(start_time=start, group_by=["project_id", "line_item"], limit=7)
    for bucket_ in page.data:
        for result in bucket_.results:
            print(bucket_.start_time, result.line_item, result.amount)

> **Observe:** the model choice moves cost 100×, the tier 2×, and caching about 27% on Luna in this example. But once a human reviews failures, Luna at 92% costs more per successful document than Sol at 99%. **Cost per successful task** is the number to put in a proposal; it is also why the next section gates routing on an eval.

## 7.11 Model routing and cascades

Route each task to the cheapest model that passes its eval. A cascade runs Luna first and escalates to Sol only when validation fails, but it can only catch errors the validator can see: **valid but wrong** outputs pass straight through. The eval decides whether the cascade is good enough.

[Theory](README.md#711-model-routing-and-cascades)

In [ ]:
EFFORT_SUPPORT = {   # from the model pages, 2026-10-03
    "gpt-6-luna": {"none", "low", "medium", "high", "xhigh", "max"},
    "gpt-6.1-sol": {"low", "medium", "high", "xhigh", "max"},
    "gpt-6-astra": {"low", "medium", "high", "xhigh", "max"},
}


def outcome(model: str, task_id: int) -> str:
    """fake: deterministic per-task outcome. Luna: 90% correct, 6% invalid (caught by validation), 4% valid-but-wrong.
    Sol: 98% correct, 2% valid-but-wrong. Replace these rates with YOUR eval results."""
    x = int(hashlib.sha256(f"{model}:{task_id}".encode()).hexdigest(), 16) % 1000 / 1000
    if model == "gpt-6-luna":
        return "correct" if x < 0.90 else "invalid" if x < 0.96 else "valid_but_wrong"
    return "correct" if x < 0.98 else "valid_but_wrong"


def run_strategy(name: str, tasks: range, validator_catches_wrong: float = 0.0) -> dict:
    cost = correct = escalated = 0
    for t in tasks:
        if name in ("luna", "sol"):
            model = "gpt-6-luna" if name == "luna" else "gpt-6.1-sol"
            cost += request_cost(model, steady)
            correct += outcome(model, t) == "correct"
            continue
        result = outcome("gpt-6-luna", t)                        # cascade: Luna first
        cost += request_cost("gpt-6-luna", steady)
        caught = result == "invalid" or (result == "valid_but_wrong" and (t % 100) / 100 < validator_catches_wrong)
        if caught:
            escalated += 1
            cost += request_cost("gpt-6.1-sol", steady)
            result = outcome("gpt-6.1-sol", t)
        correct += result == "correct"
    n = len(tasks)
    return {"strategy": name, "accuracy": correct / n, "usd_per_doc": cost / n, "escalation_rate": escalated / n}


tasks = range(5_000)
for label, row in [("Luna only", run_strategy("luna", tasks)), ("Sol only", run_strategy("sol", tasks)),
                   ("cascade Luna -> Sol", run_strategy("cascade", tasks)),
                   ("cascade + stricter validator", run_strategy("cascade", tasks, validator_catches_wrong=0.5))]:
    print(f"{label:<30} accuracy={row['accuracy']:.1%}  ${row['usd_per_doc']:.6f}/doc  escalated={row['escalation_rate']:.1%}")

In [ ]:
def check_effort(model: str, effort: str) -> None:
    if effort not in EFFORT_SUPPORT[model]:
        raise ValueError(f"{model} does not support reasoning.effort={effort!r}; options: {sorted(EFFORT_SUPPORT[model])}")


for model, effort in [("gpt-6-luna", "none"), ("gpt-6-astra", "none"), ("gpt-6.1-sol", "low")]:
    try:
        check_effort(model, effort)
        print(f"{model} effort={effort}: ok")
    except ValueError as err:
        print(err)

> **Observe:** the cascade costs a fraction of Sol-only, but its accuracy sits between the two because Luna's valid-but-wrong outputs never escalate. A stricter validator (for example a cross-field check) closes the gap at a slightly higher escalation rate. These rates are invented; with real ones, the comparison only means something if it comes from your eval. Whether the cascade ships depends on the eval's per-segment results, not on the average cost.
>
> **Try it:** set Luna's valid-but-wrong rate to 1% in `outcome` and re-run. At what rate does the cascade beat Sol on cost per successful document once a wrong answer costs $5 downstream?

## 7.12 Production readiness: setting up the account

Two production controls you can rehearse offline. First, **allowed service tiers per project**: a request that asks for, or resolves to, a disallowed tier fails with `400 invalid_request_error` and `param == "service_tier"` (`fast` is evaluated as `priority`). That is a configuration error, so the retry classifier from 7.9 must say "fix", not "retry". Second, a small **readiness linter** that applies the 7.12 checklist to a project configuration.

[Theory](README.md#712-production-readiness-setting-up-the-account)

In [ ]:
# A bulk-job project that allows only Standard and Flex. The fake returns the error-codes guide's message, type and param;
# the guide publishes no error.code for it, so the fake sends none.
for label, project_tier, tier in [("explicit service_tier='fast'", "default", "fast"),
                                  ("omitted, project tier = Fast", "fast", None),
                                  ("explicit service_tier='flex'", "default", "flex")]:
    backend = FakeOpenAI(project_tier=project_tier, allowed_tiers={"default", "flex"})
    kwargs = dict(request, **({"service_tier": tier} if tier else {}))
    try:
        r = offline_client(backend).responses.create(**kwargs)
        print(f"{label:<30} -> 200, served by {r.service_tier}")
    except APIStatusError as err:
        print(f"{label:<30} -> {err.status_code} {type(err).__name__} param={err.param!r} "
              f"action={retry_action(err.status_code, err.code, err_type=err.type)} HTTP calls={len(backend.log)}")

In [ ]:
@dataclass
class ProjectConfig:
    name: str
    environment: str                         # "staging" or "production"
    workload: str                           # "batch" or "interactive"
    allowed_tiers: set[str] | None           # None = no restriction configured
    hard_limit_usd: float | None             # None = no hard spend limit
    alert_thresholds_usd: list[float]
    key_expiry_days: int | None              # None = keys never expire
    keys_from: str                           # "env", "secret_manager" or "code"
    shared_with: list[str] = field(default_factory=list)   # other environments or workloads in the SAME project


def readiness_findings(p: ProjectConfig) -> list[str]:
    """The README 7.12 checklist as code. Thresholds such as the key lifetime are YOUR policy, not OpenAI rules."""
    findings = []
    if p.shared_with:
        findings.append(f"shares a project with {p.shared_with}: give staging/production and batch/interactive their own projects")
    if p.workload == "batch" and (p.allowed_tiers is None or "priority" in p.allowed_tiers or "fast" in p.allowed_tiers):
        findings.append("bulk project can run at Fast (2x): allow only default/flex")
    if p.hard_limit_usd is None:
        findings.append("no hard spend limit")
    elif not any(a < p.hard_limit_usd for a in p.alert_thresholds_usd):
        findings.append("no spend alert below the hard limit: humans hear about it only when traffic stops")
    if p.key_expiry_days is None:
        findings.append("API keys never expire: set an expiry date and a rotation process")
    if p.keys_from == "code":
        findings.append("API key in code: load it from the environment or a secret manager")
    return findings


projects = [
    ProjectConfig("everything", "production", "batch", None, 5_000, [], None, "code", ["staging", "interactive"]),
    ProjectConfig("nightly-scoring-prod", "production", "batch", {"default", "flex"}, 3_000, [1_500, 2_400], 90, "secret_manager"),
]
for p in projects:
    findings = readiness_findings(p)
    print(f"{p.name}: {'ready' if not findings else f'{len(findings)} finding(s)'}")
    for finding in findings:
        print("  -", finding)

> **Observe:** the explicit `fast` request and the request that **omitted** `service_tier` both failed, because the project's own tier resolves to Fast. The SDK sent exactly one HTTP request each time (it does not retry a `400`), and `retry_action` classified the error as `fix_request`. The `flex` request went through. The linter turns the checklist into something a CI job can run against your project settings export.
>
> **Try it:** add a staging project that shares nothing but has no alert below its hard limit. Then extend `readiness_findings` with one rule of your own (for example, "production must not allow `flex`, because users are waiting").

## Mini-project: a batch extraction pipeline with resubmission and a cost report

Put the chapter together on ~100 synthetic invoices:

1. Build the JSONL with deterministic `custom_id`s, the rules as a cacheable prefix with an explicit breakpoint, and a manifest that is saved with each batch's IDs as soon as the batch exists (7.2, 7.3, 7.7).
2. Run the batch lifecycle offline. The fake's "24-hour window" is short, so the last lines expire (7.2).
3. Classify every output and error line into ok / resubmit / fix first / review (7.4).
4. Resubmit only the retryable `custom_id`s as a **second batch**; fix the fix-first items and send them through **Flex with a Standard fallback**; route the rest to human review (7.4–7.6).
5. Check the manifest for missing IDs, verify the accepted values against the source documents, then print a cost report: Batch vs synchronous Standard, cache none / miss / hit, and cost per successful document including review (7.7, 7.10).

In [ ]:
rng = random.Random(42)
VENDORS = ["Acme Ltd", "Nordlicht GmbH", "Blue Fern Design", "Kato Electronics", "Harbor Cleaning Co.", "Sakura Foods KK",
           "Rio Verde SA", "Polar Freight AS", "Maple Print Inc", "Orbit Cloud LLC"]
CURRENCIES = ["EUR", "USD", "GBP", "CAD", "JPY", "NOK"]
MARKERS = {7: "TOO_LONG", 23: "TOO_LONG", 12: "[[FLAKY]]", 34: "[[FLAKY]]", 56: "[[FLAKY]]", 78: "[[FLAKY]]",
           45: "[[RAMBLING]]", 93: "[[RAMBLING]]", 5: "[[SMUDGED]]", 50: "[[SMUDGED]]", 95: "[[SMUDGED]]"}

corpus = {}
for i in range(100):
    doc = (f"Vendor: {rng.choice(VENDORS)} | Date: 2026-{rng.randint(1, 9):02d}-{rng.randint(1, 28):02d} | "
           f"Currency: {rng.choice(CURRENCIES)} | Total: {rng.randint(10, 5000)}.{rng.randint(0, 99):02d}")
    marker = MARKERS.get(i, "")
    if marker == "TOO_LONG":
        doc += " [[APPENDIX]] " + "General terms and conditions apply. " * 400    # a scanned T&C appendix
    elif marker:
        doc += f" {marker}"
    corpus[f"inv-{i:06d}"] = doc

MP_RULES = RULES                             # ~1,500 tokens of rules + worked examples, identical on every line
PROMPT_VERSION = "invoice-v3"


def mp_line(custom_id: str, document: str, *, max_output_tokens: int = 2000) -> dict:
    """The 7.7 cache layout inside a batch line: the rules sit in a developer input_text block with an explicit breakpoint
    (top-level instructions cannot hold one), explicit-only mode so the changing document is never written."""
    line = batch_line(custom_id, document, max_output_tokens=max_output_tokens)
    body = line["body"]
    del body["instructions"]
    body["prompt_cache_options"] = {"mode": "explicit"}
    body["input"] = [
        {"role": "developer", "content": [{"type": "input_text", "text": MP_RULES,
                                           "prompt_cache_breakpoint": {"mode": "explicit"}}]},
        {"role": "user", "content": f"<document>\n{document}\n</document>"},
    ]
    return line


mp_lines = [mp_line(make_custom_id(doc_id, PROMPT_VERSION), text) for doc_id, text in corpus.items()]
assert all(not lint_line(line) for line in mp_lines)
mp_manifest = Manifest(job="invoices-2026-10-03", model=MODEL, prompt_version=PROMPT_VERSION,
                       custom_ids=[line["custom_id"] for line in mp_lines])
print(len(mp_lines), "lines,", sum(len(json.dumps(l)) + 1 for l in mp_lines) // 1024, "KiB; manifest:", mp_manifest.save().name)

In [ ]:
mp_backend = FakeOpenAI(lines_per_poll=30, window_polls=3, seed=3)      # 90 lines fit in the "24 h" window
mp_api = offline_client(mp_backend)
b1, out1, err1 = run_batch(mp_api, mp_lines, metadata={"job": mp_manifest.job, "prompt_version": PROMPT_VERSION},
                           on_created=mp_manifest.record)              # IDs saved to disk before the first poll
print("manifest on disk:", json.loads((WORKDIR / f"{mp_manifest.job}.manifest.json").read_text())["batches"])

billed = []                                   # (usage, tier) for every billed 200 response
final: dict[str, Invoice] = {}
pending: dict[str, str] = {}                  # custom_id -> latest failure kind


def absorb(lines_: list[dict], tier: str) -> None:
    for line in lines_:
        kind, invoice = classify(line)
        response = line.get("response") or {}
        if response.get("status_code") == 200:
            billed.append((ResponseUsage.model_validate(response["body"]["usage"]), tier))
        if kind == "ok":
            final[line["custom_id"]] = invoice
            pending.pop(line["custom_id"], None)
        else:
            pending[line["custom_id"]] = kind


absorb(out1 + err1, "batch")
print("round 1:", len(final), "ok |", Counter(bucket(k) for k in pending.values()), "|", Counter(pending.values()))

In [ ]:
# Round 2: resubmit the retryable lines unchanged, as a second batch (same custom_ids, rebuilt from the manifest)
resubmit = sorted(cid for cid, kind in pending.items() if bucket(kind) == "resubmit")
b2, out2, err2 = run_batch(mp_api, [mp_line(cid, corpus[doc_id_of(cid)]) for cid in resubmit],
                           metadata={"job": mp_manifest.job + "-retry1", "prompt_version": PROMPT_VERSION}, verbose=False,
                           on_created=lambda b: mp_manifest.record(b, role="retry1"))
absorb(out2 + err2, "batch")
print(f"round 2: resubmitted {len(resubmit)} -> {len(final)} ok |", Counter(pending.values()))
print("batches in the manifest:", [(b["role"], b["batch_id"]) for b in mp_manifest.batches])

In [ ]:
# Round 3: fix first, then run synchronously on Flex (deadline is close) with a Standard fallback
def fixed_request(cid: str, kind: str) -> dict:
    doc = corpus[doc_id_of(cid)]
    max_out = 2000
    if kind == "http_400":
        doc = doc.split(" [[APPENDIX]]")[0]          # drop the boilerplate appendix that blew the context window
    if kind.startswith("incomplete"):
        max_out = 8000                               # the output was cut off: raise the limit
    return mp_line(cid, doc, max_output_tokens=max_out)["body"]


fix_ids = sorted(cid for cid, kind in pending.items() if bucket(kind) == "fix_first")
for cid in fix_ids:
    kind = pending[cid]
    r = create_flex_with_fallback(mp_api, deadline_s=60, standard_reserve_s=10, base_delay=0.01, **fixed_request(cid, kind))
    absorb([{"custom_id": cid, "response": {"status_code": 200, "body": r.model_dump(mode="json")}, "error": None}],
           r.service_tier or "default")
    print(f"  {cid}: {kind:<28} -> {'ok' if cid in final else pending[cid]} (served by {r.service_tier})")

review_queue = {cid: kind for cid, kind in pending.items()}
missing = set(mp_manifest.custom_ids) - set(final) - set(review_queue)      # every submitted ID must end somewhere
print("ok:", len(final), "| human review:", review_queue, "| missing:", missing or "none")


# Accepted is not the same as correct: compare every accepted invoice with its source document (an eval in miniature).
# A pipeline that only checks the schema would happily accept values copied from the worked examples in the rules.
def matches_source(cid: str, inv: Invoice) -> bool:
    src = corpus[doc_id_of(cid)]
    return (f"Vendor: {inv.vendor} |" in src and f"Date: {inv.invoice_date} |" in src
            and f"Currency: {inv.currency} |" in src and f"Total: {inv.total:.2f}" in src)


mismatched = sorted(cid for cid, inv in final.items() if not matches_source(cid, inv))
print("accepted but not matching the source:", mismatched or "none")
assert not mismatched, "the extractor returned values that are not in the document"

In [ ]:
def what_if(usages: list[tuple[ResponseUsage, str]], *, tier: str | None, cache: str) -> float:
    """Re-price the SAME tokens under another tier and a cache outcome for the rules prefix:
    "none" = no breakpoint at all (explicit-only mode, nothing written or read); "miss" = every line writes the prefix
    (1.25x) and none reads it, the pessimistic plan; "hit" = the first line writes, every later line reads (0.1x).
    OpenAI promises no hit rate inside a batch: price with "miss", treat hits as a bonus."""
    total = 0.0
    for i, (u, served) in enumerate(usages):
        if cache != "none":
            written = prefix if cache == "miss" or i == 0 else 0
            u = usage_of(u.input_tokens, u.output_tokens, cached=0 if written else prefix, written=written,
                         reasoning=u.output_tokens_details.reasoning_tokens)
        total += request_cost(MODEL, u, tier or served)
    return total


prefix = rough_tokens(MP_RULES)
REVIEW_COST = 0.50                              # assumed $ per human review
succeeded = len(final) + len(review_queue)      # reviewed documents end correct too (see cost_per_successful_task, 7.10)
rows = {}
for tier_label, tier in [("as run (Batch + Flex)", None), ("all synchronous Standard", "default")]:
    for cache in ["none", "miss", "hit"]:
        rows[f"{tier_label}, cache {cache}"] = what_if(billed, tier=tier, cache=cache)
print(f"{'scenario':<38} {'token cost':>11} {'per auto-accepted':>18} {'per success incl. review':>25}")
for name, usd in rows.items():
    print(f"{name:<38} ${usd:>10.5f} ${usd / len(final):>17.7f} ${(usd + REVIEW_COST * len(review_queue)) / succeeded:>24.5f}")
print(f"\nbilled responses: {len(billed)} for {len(corpus)} documents; auto-accepted {len(final)}, reviewed {len(review_queue)}")

> **Observe:**
> - Round 1 expired the last 10 lines (the fake's window) and failed a handful more; only the retryable ones went into the second batch, and the `[[FLAKY]]` and expired lines then succeeded unchanged.
> - The oversized and truncated documents needed a **changed** request; resubmitting them as-is would have failed again. Flex handled them at batch prices without waiting another "24 hours".
> - Token costs are tiny next to **human review**: three smudged invoices cost more to review than all 100 extractions. The cheapest optimization here is a better prompt or validator for currency symbols, not a cheaper tier.
> - The extracted values were checked against the source documents: without that check, a fake (or a real model) that copies a worked example from the rules would pass the schema and every business rule.
> - The lines put the rules in a `developer` block with an explicit breakpoint, so a cache hit is *possible* inside the batch. "cache hit" is the best case and "cache miss" (every line pays the 1.25× write) the pessimistic one; with no promised hit rate, plan with the miss row and treat hits as a bonus. Without a breakpoint (implicit mode) every line would write rules + document and none would be read.
> - "Per success incl. review" divides by the auto-accepted **and** the reviewed documents, because human review turns those into successes too (the same rule as `cost_per_successful_task` in 7.10).

> **Try it:**
> - Move the 10 Sol-worthy documents to `gpt-6.1-sol`: you now need two input files (one model per file). Extend the pipeline and the cost report.
> - Add a retry-with-feedback step (chapter 06) for the `rule_failed:currency` cases before they reach a human, and recompute cost per successful document.

In [ ]:
# (live, optional, ~$0.001) Submit a real 3-line batch. It completes within 24 hours; check it with the next cell later.
live_lines = [batch_line(make_custom_id(doc_id, "invoice-v3"), docs[doc_id]) for doc_id in list(docs)[:3]]
assert all(not lint_line(line) for line in live_lines)
path = write_jsonl(WORKDIR / "live-3.jsonl", live_lines)
with open(path, "rb") as f:
    live_file = client.files.create(file=f, purpose="batch")
live_batch = client.batches.create(input_file_id=live_file.id, endpoint="/v1/responses", completion_window="24h",
                                   metadata={"job": "oa-ch07-live-demo", "prompt_version": "invoice-v3"})
LIVE_MARKER.write_text(live_batch.id)                                   # remembered across kernel restarts
print(live_batch.id, live_batch.status)

In [ ]:
# (live) Check the batch from the previous cell (safe to run any time; does nothing if you never submitted one)
if not LIVE_MARKER.exists():
    print("no live batch submitted yet")
else:
    live = client.batches.retrieve(LIVE_MARKER.read_text().strip())
    print(live.id, live.status, live.request_counts)
    if live.status in TERMINAL:
        for line in read_jsonl(client, live.output_file_id) + read_jsonl(client, live.error_file_id):
            print(" ", line["custom_id"], classify(line))

## Recap

- Cost = tokens × price × tier (× region), and reasoning tokens are output tokens. Model choice moves cost 100×, tier 2×, caching tens of percent.
- Batch: JSONL with one model per file, `purpose="batch"`, `completion_window="24h"`, poll or webhook, then two files. 50% off, separate limits, 24-hour worst case.
- `custom_id` is a designed primary key; keep a manifest; join by ID, never by order; resubmit only retryable failures and fix the rest first.
- Plan deadlines on the worst case plus the retry path. Flex is batch pricing on a sync call (handle `429`, fall back); Fast is 2× for latency; log the **served** tier.
- On GPT-6, implicit caching can write without ever reading: put an explicit breakpoint after the stable prefix, in a developer `input_text` block.
- Plan TPM with your `max_output_tokens`; classify errors by status and `error.code`; never nest your retries over the SDK's.
- Report cost per successful task, and gate model routing on an eval.
- Production setup: separate staging and production projects, allowed service tiers per project (a wrong tier is a `400` to fix, not retry), keys with an expiry, and spend alerts below hard limits.

Next: [homework](02_homework.ipynb)